In [1]:
# Install condacolab to handle Miniconda installation in Google Colab
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:06
🔁 Restarting kernel...


In [1]:
# Install Montreal Forced Aligner (will be installed in a dedicated environment later)
!conda clean --all -y # Clean up any existing conda caches
!conda update -n base -c conda-forge conda -y # Update conda in the base environment to ensure optimal dependency resolution

# Install Python audio libraries and Gemini SDK
import sys # Import sys to get the correct Python executable
!{sys.executable} -m pip install transformers accelerate librosa soundfile praat-parselmouth google-generativeai textgrid

Will remove 84 (59.1 MB) tarball(s).
Will remove 1 index cache(s).
Will remove 6 (593 KB) package(s).
There are no tempfile(s) to remove.
There are no logfile(s) to remove.
Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - conda


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.6.17  |       hbd8a1cb_0         126 KB  conda-forge
    certifi-2026.6.17          |     pyhd8ed1ab_0         131 KB  conda-forge
    conda-24.11.3              |  py311h38be061_0         1.1 MB  conda-forge
    openssl-3.6.3              |       h35e630c_0         3.0 MB  conda-forge
    ------------------------------------------------------------
                                           Total:         4.4 MB

The following packages will be UPDATED:

  ca-certificates    

In [2]:
import os

zip_file_path = "/content/js_new.zip"
extract_directory = "/content/"

# Create the extraction directory if it doesn't exist
if not os.path.exists(extract_directory):
    os.makedirs(extract_directory)

# Unzip the file using a shell command
!unzip -o "{zip_file_path}" -d "{extract_directory}"

print(f"Successfully unzipped '{zip_file_path}' to '{extract_directory}'")

Archive:  /content/js_new.zip
   creating: /content/js_new/
  inflating: /content/__MACOSX/._js_new  
  inflating: /content/js_new/j-coo4-m5.wav  
  inflating: /content/__MACOSX/js_new/._j-coo4-m5.wav  
  inflating: /content/js_new/j-dec5-m4.wav  
  inflating: /content/__MACOSX/js_new/._j-dec5-m4.wav  
  inflating: /content/js_new/j-whq2-m5.wav  
  inflating: /content/__MACOSX/js_new/._j-whq2-m5.wav  
  inflating: /content/js_new/j-yno3-f2.wav  
  inflating: /content/__MACOSX/js_new/._j-yno3-f2.wav  
  inflating: /content/js_new/j-dec1-f7.wav  
  inflating: /content/__MACOSX/js_new/._j-dec1-f7.wav  
  inflating: /content/js_new/j-coo2-f3.wav  
  inflating: /content/__MACOSX/js_new/._j-coo2-f3.wav  
  inflating: /content/js_new/j-yno1-f7.wav  
  inflating: /content/__MACOSX/js_new/._j-yno1-f7.wav  
  inflating: /content/js_new/j-dec3-f2.wav  
  inflating: /content/__MACOSX/js_new/._j-dec3-f2.wav  
  inflating: /content/js_new/j-dqu3-m2.wav  
  inflating: /content/__MACOSX/js_new/._j-dqu

In [3]:
import os

zip_file_path = "/content/js_new_transcripts.zip"
extract_directory = "/content/"

# Create the extraction directory if it doesn't exist
if not os.path.exists(extract_directory):
    os.makedirs(extract_directory)

# Unzip the file using a shell command
!unzip -o "{zip_file_path}" -d "{extract_directory}"

print(f"Successfully unzipped '{zip_file_path}' to '{extract_directory}'")

Archive:  /content/js_new_transcripts.zip
   creating: /content/js_new_transcripts/
  inflating: /content/js_new_transcripts/j-dqu1-m4.txt  
  inflating: /content/js_new_transcripts/j-dec1-f4.txt  
  inflating: /content/js_new_transcripts/j-yno3-f1.txt  
  inflating: /content/js_new_transcripts/j-dec3-f1.txt  
  inflating: /content/js_new_transcripts/j-yno1-f4.txt  
  inflating: /content/js_new_transcripts/j-dqu3-m1.txt  
  inflating: /content/js_new_transcripts/j-dec7-m2.txt  
  inflating: /content/js_new_transcripts/j-dec7-m3.txt  
  inflating: /content/js_new_transcripts/j-coo2-f1.txt  
  inflating: /content/js_new_transcripts/j-yno1-f5.txt  
  inflating: /content/js_new_transcripts/j-dec1-f5.txt  
  inflating: /content/js_new_transcripts/j-dqu1-m5.txt  
  inflating: /content/js_new_transcripts/j-dec5-m4.txt  
  inflating: /content/js_new_transcripts/j-coo4-m5.txt  
  inflating: /content/js_new_transcripts/j-yno3-f2.txt  
  inflating: /content/js_new_transcripts/j-whq2-m5.txt  
  in

In [4]:
# ==============================================================================
# STEP 2: PREPARE DIRECTORIES FOR MFA
# ==============================================================================
from pathlib import Path
import shutil
import os
import librosa
import soundfile as sf

# Define your source folders (Assumes they are already unzipped in /content/)
AUDIO_SOURCE_DIR = Path("/content/js_new")
TEXT_SOURCE_DIR = Path("/content/js_new_transcripts")

# MFA requires audio and text to be in the exact same directory
MFA_INPUT_DIR = Path("/content/mfa_input")
MFA_OUTPUT_DIR = Path("/content/mfa_output")

MFA_INPUT_DIR.mkdir(parents=True, exist_ok=True)
MFA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Standardizing audio and copying to MFA staging directory...")

# 1. Process and move audio files (Resample to 16kHz, Mono)
for audio_file in AUDIO_SOURCE_DIR.glob("*"):
    if audio_file.suffix.lower() in [".wav", ".mp3", ".flac", ".m4a"]:
        audio, sr = librosa.load(audio_file, sr=16000, mono=True)
        output_file = MFA_INPUT_DIR / f"{audio_file.stem}.wav"
        sf.write(output_file, audio, 16000, subtype="PCM_16")

# 2. Move transcript text files
for text_file in TEXT_SOURCE_DIR.glob("*.txt"):
    shutil.copy(str(text_file), str(MFA_INPUT_DIR / text_file.name))

print(f"Preparation complete. {len(list(MFA_INPUT_DIR.glob('*.wav')))} audio files ready for alignment.")

Standardizing audio and copying to MFA staging directory...
Preparation complete. 262 audio files ready for alignment.


In [5]:
%%bash
eval "$(conda shell.bash hook)"

# Remove any existing 'mfa_env' to ensure a clean slate
conda env remove -n mfa_env -y

# Create a new conda environment named 'mfa_env' with Python 3.9 (trying a different Python version for compatibility)
conda create -n mfa_env python=3.9 -y

# Set MPLBACKEND to a non-interactive backend to avoid issues with matplotlib in Colab
export MPLBACKEND=Agg

# Activate the new 'mfa_env'
# Note: `conda activate` alone may not fully persist in non-interactive bash.
# For subsequent commands, use `conda run -n mfa_env` explicitly.
conda activate mfa_env

# Install montreal-forced-aligner into the 'mfa_env'
conda install -c conda-forge montreal-forced-aligner -y

echo "Verifying MFA installation in mfa_env:"
conda run -n mfa_env mfa --version

# Download the standard US English acoustic model and dictionary
conda run -n mfa_env mfa model download dictionary english_us_arpa
conda run -n mfa_env mfa model download acoustic english_us_arpa

Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local/envs/mfa_env

  added / updated specs:
    - python=3.9


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.8.1             |       hecca717_1          76 KB  conda-forge
    libffi-3.5.2               |       h3435931_0          57 KB  conda-forge
    libgcc-15.2.0              |      he0feb66_19        1017 KB  conda-forge
    libgcc-ng-15.2.0           |      h69a702a_19          27 KB  conda-forge
    libgomp-15.2.0             |      he0feb66_19         590 KB  conda-forge
    liblzma-5.8.3              


EnvironmentLocationNotFound: Not a conda environment: /usr/local/envs/mfa_env



==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda




==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda


                                                                                
 Usage: mfa [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 Try 'mfa --help' for help                                                      
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such option: --version                                                    │
╰───────────────────────────────────────────────────

In [6]:
%%bash
export MPLBACKEND=Agg
conda run -n mfa_env mfa align /content/mfa_input english_us_arpa english_us_arpa /content/mfa_output/

 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262/100  [ 0:00:02 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262/262  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 261/262  [ 0:00:11 < 0:00:01 , 71 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 261/262  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 261/262  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 261/262  [ 0:00:03 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 261/262  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 261/262  [ 0:00:02 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 261/262  [ 0:00:02 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 261/262  [ 0:00:02 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 261/262  [ 0:00:00 < 0:00:01 , 2,852 it/s ]



 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 INFO     Found 1 speaker across 262 files, average number of utterances per    
          speaker: 262.0                                                        
 INFO     Initializing multiprocessing jobs...                                  
 WARNING  Number of jobs was specified as 3, but due to only having 1 speakers, 
          MFA will only use 1 jobs. Use the --single_speaker flag if you would  
          like to split utterances across jobs regardless of their speaker.     
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs...                                                   
 INFO     Calculating CMVN...                                                   
 INFO     Generating final features...                                          
 WARNING  There were 1 utter

In [8]:
# ==============================================================================
# STEP 4: EXTRACT ACOUSTIC FEATURES & INJECT IVIE METADATA
# ==============================================================================
import parselmouth
import textgrid
import numpy as np
import re

all_blueprints = {}

VARIETY_MAP = {'b': 'Belfast', 'c': 'Cambridge', 'd': 'Dublin', 'l': 'Leeds', 'n': 'Newcastle', 'p': 'Bradford', 'j': 'London'}

def get_sentence_category(type_code):
    mapping = {
        "sta": "Simple Statement", "dec": "Simple Statement",
        "que": "Morphosyntactic Question", "dqu": "Morphosyntactic Question",
        "inv": "Inversion Question", "yno": "Inversion Question",
        "whq": "WH-Question",
        "coo": "Coordination Structure"
    }
    return mapping.get(type_code.lower(), "Unknown Sentence Type")

for tg_file in sorted(os.listdir(MFA_OUTPUT_DIR)):
    if not tg_file.endswith(".TextGrid"):
        continue

    base_name = os.path.splitext(tg_file)[0]
    wav_file = os.path.join(MFA_INPUT_DIR, f"{base_name}.wav")

    if not os.path.exists(wav_file):
        continue

    # 1. Parse IViE Metadata from filename (e.g., b-coo1-f1)
    region, gender, category = "Unknown", "Unknown", "Unknown"
    match = re.search(r'([a-z])-([a-z]{3})\d+-([fm])\d+', base_name)
    if match:
        region = VARIETY_MAP.get(match.group(1), "Unknown")
        category = get_sentence_category(match.group(2))
        gender = "Female" if match.group(3) == 'f' else "Male"

    # 2. Load Praat and TextGrid
    tg = textgrid.TextGrid.fromFile(os.path.join(MFA_OUTPUT_DIR, tg_file))
    snd = parselmouth.Sound(wav_file)
    pitch, intensity = snd.to_pitch(), snd.to_intensity()

    global_pitch_mean = np.nanmean(pitch.selected_array["frequency"])
    global_intensity_mean = np.nanmean(intensity.values)

    blueprint_lines = [
        f"FILE: {base_name}",
        f"Dataset Context: IViE Corpus (UK Regional Dialects)",
        f"Speaker Region/Variety: {region}",
        f"Speaker Gender: {gender}",
        f"Sentence Type Category: {category}",
        f"Global Audio Baselines: Pitch = {global_pitch_mean:.1f}Hz, Intensity = {global_intensity_mean:.1f}dB\n"
    ]

    word_tier = next((t for t in tg.tiers if t.name.lower() == "words"), None)
    if not word_tier: continue

    # 3. Extract word-level metrics
    for interval in word_tier:
        if not interval.mark.strip(): continue
        start, end = interval.minTime, interval.maxTime

        try:
            word_snd = snd.extract_part(from_time=start, to_time=end)
            w_pitch, w_intensity = word_snd.to_pitch(), word_snd.to_intensity()

            p_vals = w_pitch.selected_array["frequency"]
            w_pitch_val = np.mean(p_vals[p_vals > 0]) if len(p_vals[p_vals > 0]) > 0 else np.nan
            w_int_val = np.nanmax(w_intensity.values) if w_intensity.values.size > 0 else np.nan
        except:
            w_pitch_val, w_int_val = np.nan, np.nan

        p_delta = w_pitch_val - global_pitch_mean
        i_delta = w_int_val - global_intensity_mean

        blueprint_lines.append(
            f"Word '{interval.mark}': Duration={end-start:.2f}s, "
            f"Pitch={w_pitch_val:.1f}Hz ({'+' if p_delta > 0 else ''}{p_delta:.1f}Hz vs baseline), "
            f"Intensity={w_int_val:.1f}dB ({'+' if i_delta > 0 else ''}{i_delta:.1f}dB vs baseline)"
        )

    all_blueprints[base_name] = "\n".join(blueprint_lines)

print(f"Successfully generated extracted acoustic blueprints for {len(all_blueprints)} files.")

Successfully generated extracted acoustic blueprints for 261 files.


In [10]:
# ==============================================================================
# STEP 5: GEMMA 3 LOCAL INFERENCE (SINGLE INFERENCE PER INPUT)
# ==============================================================================
from huggingface_hub import login
from google.colab import userdata
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json

hf_token = userdata.get('HF_TOKEN')
login(hf_token)

model_id = "google/gemma-3-27b-it"
print(f"Loading {model_id} in bfloat16. This will take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

Loading google/gemma-3-27b-it in bfloat16. This will take a few minutes...


config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/127k [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [11]:
print("\n--- BEGINNING SEQUENTIAL PROSODY EVALUATION ---")

results_dict = {}

# Iterate one by one through the files
for base_name, blueprint in all_blueprints.items():
    print(f"Evaluating {base_name}...")

    prompt = f"""You are an expert phonetician and strict JSON data generator.
Analyze the following acoustic event sequence of a spoken sentence and evaluate the prosody.

INPUT DATA:
{blueprint}

TASK:
1. Identify the 'predicted_intent' based on the acoustic features (e.g., Statement, Question, Surprise, Choice).
2. Identify which word(s) receive prosodic emphasis based on spikes in duration, pitch, or intensity compared to the baseline. Return them as a list of strings.
3. Determine the 'expressiveness_pattern' (e.g., Rising, Falling, Flat, Peaking).

ABSOLUTE RULES:
- You MUST output ONLY valid JSON.
- Do NOT wrap the JSON in Markdown formatting blocks (e.g., no ```json).
- Do NOT include any explanations, introductory text, or concluding text.

REQUIRED JSON SCHEMA:
{{
  "predicted_intent": "string",
  "predicted_emphasis_words": ["string", "string"],
  "expressiveness_pattern": "string"
}}
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.1, # Low temperature for strict structural compliance
        do_sample=True
    )

    input_length = inputs["input_ids"].shape[1]
    response_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

    # Store the raw text output
    results_dict[base_name] = response_text

    # Print it to Colab console to monitor progress
    print(response_text)
    print("-" * 50)

# Optional: Save all predictions to a final JSON file for grading
with open("/content/llm_predictions.json", "w") as f:
    json.dump(results_dict, f, indent=4)

print("Inference Complete! Results saved to /content/llm_predictions.json")


--- BEGINNING SEQUENTIAL PROSODY EVALUATION ---
Evaluating j-coo1-f1...
```json
{
  "predicted_intent": "Choice",
  "predicted_emphasis_words": ["you", "limes", "lemons"],
  "expressiveness_pattern": "Peaking"
}
```
--------------------------------------------------
Evaluating j-coo1-f2...
```json
{
  "predicted_intent": "Choice",
  "predicted_emphasis_words": ["you", "growing", "lemons"],
  "expressiveness_pattern": "Peaking"
}
```
--------------------------------------------------
Evaluating j-coo1-f3...
```json
{
  "predicted_intent": "Choice",
  "predicted_emphasis_words": ["you", "growing", "limes", "lemons"],
  "expressiveness_pattern": "Peaking"
}
```
--------------------------------------------------
Evaluating j-coo1-f4...
```json
{
  "predicted_intent": "Choice",
  "predicted_emphasis_words": ["limes", "or"],
  "expressiveness_pattern": "Peaking"
}
```
--------------------------------------------------
Evaluating j-coo1-f5...
```json
{
  "predicted_intent": "Choice",
  "pred

In [12]:
%%bash
zip -r /content/mfa_output.zip /content/mfa_output
echo "Successfully zipped /content/mfa_output to /content/mfa_output.zip"

  adding: content/mfa_output/ (stored 0%)
  adding: content/mfa_output/j-coo4-m5.TextGrid (deflated 86%)
  adding: content/mfa_output/j-coo5-f7.TextGrid (deflated 84%)
  adding: content/mfa_output/j-dec7-m4.TextGrid (deflated 83%)
  adding: content/mfa_output/j-dec1-f2.TextGrid (deflated 83%)
  adding: content/mfa_output/j-dqu1-f4.TextGrid (deflated 83%)
  adding: content/mfa_output/j-dec1-m5.TextGrid (deflated 84%)
  adding: content/mfa_output/j-coo3-m2.TextGrid (deflated 84%)
  adding: content/mfa_output/j-coo3-m4.TextGrid (deflated 84%)
  adding: content/mfa_output/j-coo3-f6.TextGrid (deflated 84%)
  adding: content/mfa_output/j-coo1-m2.TextGrid (deflated 85%)
  adding: content/mfa_output/j-dec2-f1.TextGrid (deflated 84%)
  adding: content/mfa_output/j-coo4-f2.TextGrid (deflated 85%)
  adding: content/mfa_output/j-dqu3-f1.TextGrid (deflated 83%)
  adding: content/mfa_output/j-yno2-m4.TextGrid (deflated 84%)
  adding: content/mfa_output/j-yno3-f1.TextGrid (deflated 84%)
  adding: con

In [13]:
all_blueprints

{'j-coo1-f1': "FILE: j-coo1-f1\nDataset Context: IViE Corpus (UK Regional Dialects)\nSpeaker Region/Variety: London\nSpeaker Gender: Female\nSentence Type Category: Coordination Structure\nGlobal Audio Baselines: Pitch = 154.2Hz, Intensity = 74.0dB\n\nWord 'are': Duration=0.11s, Pitch=232.9Hz (+78.7Hz vs baseline), Intensity=81.7dB (+7.7dB vs baseline)\nWord 'you': Duration=0.13s, Pitch=256.6Hz (+102.4Hz vs baseline), Intensity=82.1dB (+8.1dB vs baseline)\nWord 'growing': Duration=0.42s, Pitch=217.5Hz (+63.3Hz vs baseline), Intensity=82.4dB (+8.4dB vs baseline)\nWord 'limes': Duration=0.47s, Pitch=182.1Hz (+27.9Hz vs baseline), Intensity=84.3dB (+10.2dB vs baseline)\nWord 'or': Duration=0.15s, Pitch=99.4Hz (-54.8Hz vs baseline), Intensity=81.5dB (+7.5dB vs baseline)\nWord 'lemons': Duration=0.65s, Pitch=181.3Hz (+27.1Hz vs baseline), Intensity=82.9dB (+8.9dB vs baseline)",
 'j-coo1-f2': "FILE: j-coo1-f2\nDataset Context: IViE Corpus (UK Regional Dialects)\nSpeaker Region/Variety: Londo